# 06 - Attribute-Based Content Recommendation Model
## E-Commerce Recommendation System

**Objective:** Vectorize anonymized item properties with TF-IDF and compute sparse cosine similarity recommendations.

In [2]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
import sys
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.content_based import ContentBasedRecommender

# Load sample of item properties for content modeling
p1 = pd.read_csv(ROOT / 'dataset/item_properties_part1.csv', nrows=200000)
user_item = pd.read_parquet(ROOT / 'data/processed/user_item_interactions.parquet')

# Filter user_item to items present in properties sample to prevent RAM exhaustion
p1_items = set(p1['itemid'].unique())
user_item_filtered = user_item[user_item['itemid'].isin(p1_items)]

content_model = ContentBasedRecommender(max_features=5000, min_df=2)
content_model.fit(p1, user_item_filtered)


2026-11-20 03:01:50,193 [INFO] Building content attribute tokens for 200000 property records...
2026-11-20 03:01:52,770 [INFO] Vectorizing attribute profiles for 154592 items using TF-IDF...
2026-11-20 03:01:53,634 [INFO] Content TF-IDF Matrix built: 154592 items x 5000 features | Non-zeros: 388840
2026-11-20 03:01:53,635 [INFO] Caching user history for content recommendations across 920828 records...
2026-11-20 03:01:54,283 [INFO] Cached interaction history for 659739 visitors.


In [3]:
sample_user = user_item['visitorid'].iloc[5]
recs = content_model.recommend(sample_user, top_n=10)
print(f'Content-based attribute similarity recommendations for Visitor {sample_user}:')
for r, (iid, score) in enumerate(recs, 1):
    print(f'  Rank {r:2d}: Item {iid} (Cosine similarity: {score:.4f})')


Content-based attribute similarity recommendations for Visitor 2:
